In [ ]:
# !pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


In [ ]:
# !git clone https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4
# !git switch gavriel-thesis

Cloning into 'topic_modeling_KBMI4'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 302 (delta 39), reused 51 (delta 19), pack-reused 220 (from 2)
Receiving objects: 100% (302/302), 57.66 MiB | 11.78 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Filtering content: 100% (4/4), 294.40 MiB | 16.22 MiB/s, done.
/content/topic_modeling_KBMI4
Updating files: 100% (41/41), done.
Filtering content: 100% (9/9), 1.07 GiB | 15.14 MiB/s, done.
Branch 'gavriel-thesis' set up to track remote branch 'gavriel-thesis' from 'origin'.
Switched to a new branch 'gavriel-thesis'


In [11]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from umap import UMAP
from hdbscan import HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce RTX 3060


In [5]:
df = pd.read_csv("data/preprocessed_data.csv")

df.head()

,reviewId,bank,score,year,text
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,BCAMOBILE_REVIEWS,1,2025,ribet banget nih apk sumpah dikir verif dikit ...
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,BCAMOBILE_REVIEWS,2,2025,kenapa qris enggak bisa di pakai ya daritadi l...
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,BCAMOBILE_REVIEWS,1,2025,aplikasi nya sampah kenapa tiba tiba keluar te...
3,32d28ac6-c538-4749-969f-8af060915d96,BCAMOBILE_REVIEWS,3,2025,bagus
4,f99259b7-0d45-4422-b667-6c4fd521638b,BCAMOBILE_REVIEWS,1,2025,tidak ada solusi ketika ada kendala di persuli...


In [6]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 193,827


# IndoBERT

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

e:\anaconda\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gavri\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downloa

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [ ]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [ ]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

100%|██████████| 6058/6058 [26:05<00:00,  3.87it/s]


In [ ]:
print(embeddings.shape)

(193827, 768)


In [ ]:
embeddings[0]

array([ 1.32068610e+00,  3.98074418e-01, -6.73392862e-02,  6.81021631e-01,
       -1.96820974e-01,  1.08589363e+00, -8.56553018e-01,  2.23273388e-03,
        8.79841328e-01,  3.65444899e-01, -3.26138228e-01,  4.72277194e-01,
       -8.05487573e-01,  1.67667508e-01, -2.56380171e-01, -4.40765619e-01,
       -4.77702200e-01,  2.61944145e-01, -2.86765069e-01, -2.72093844e-02,
        7.49712348e-01,  1.74879134e-02,  2.77186837e-02, -1.01489611e-01,
       -5.50755799e-01, -4.60477501e-01,  3.93649340e-01,  8.93991292e-01,
       -5.85758351e-02, -1.70096517e-01, -2.30777428e-01,  5.49784362e-01,
        2.69129515e-01,  5.88295817e-01, -7.31480896e-01,  9.41245556e-01,
        4.00836855e-01,  1.14819241e+00, -5.05113244e-01, -9.31886211e-02,
       -1.09638429e+00,  7.68689096e-01, -1.85003400e-01,  1.06485210e-01,
       -8.93403411e-01,  6.15286827e-01,  5.15412211e-01,  8.41604114e-01,
       -5.12369573e-01, -3.84789519e-02, -7.70254806e-02,  3.15342516e-01,
        1.43759996e-01,  

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.276495
Average Norm : 18.317768
Std Norm : 2.1703527


In [ ]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [ ]:
np.save(
    "indobert_embeddings.npy",
    embeddings
)

# BERTopic

In [30]:
embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (193827, 768)


In [31]:
vectorizer_model = CountVectorizer(ngram_range=(1,2))

baseline UMAP for testing purpose

In [32]:
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [33]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    prediction_data=True
)

In [34]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [35]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-05 20:23:42,216 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-05 20:31:33,396 - BERTopic - Dimensionality - Completed ✓
2026-08-05 20:31:33,399 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-05 20:31:55,041 - BERTopic - Cluster - Completed ✓
2026-08-05 20:31:55,069 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-05 20:32:02,215 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [36]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,125526,-1_bisa_enggak_di_aplikasi,"[bisa, enggak, di, aplikasi, enggak bisa, suda...",[setiap mau login selalu muncul tulisan layana...
1,0,16144,0_saldo_uang_saya_tapi,"[saldo, uang, saya, tapi, ada, biaya, ke, poto...",[saya melakukan transaksi gagal tapi saldo say...
2,1,3854,1_jaringan_sinyal_merah_bagus,"[jaringan, sinyal, merah, bagus, koneksi, indi...",[sering banget error indikator merah terus pad...
3,2,3491,2_wajah_verifikasi wajah_verifikasi_gagal,"[wajah, verifikasi wajah, verifikasi, gagal, w...","[verifikasi wajah gagal terus enggak jelas, ve..."
4,3,3369,3_tolong_perbaiki_di perbaiki_tolong di,"[tolong, perbaiki, di perbaiki, tolong di, moh...",[semenjak update aplikasi nya enggak bisa di b...
5,4,2493,4_malam_jam_00_tengah malam,"[malam, jam, 00, tengah malam, tengah, mainten...",[kalo sudah jam 23 00 sampai jam 03 00 enggak ...
6,5,2336,5_kenapa_bisa di_di buka_kenapa aplikasi,"[kenapa, bisa di, di buka, kenapa aplikasi, di...",[setelah di update aplikasi enggak bisa di buk...
7,6,2056,6_username_benar_password_sudah benar,"[username, benar, password, sudah benar, salah...",[enggak bisa login padahal username password s...
8,7,1679,7_uninstall_tetap_install_instal,"[uninstall, tetap, install, instal, ulang, uni...",[aplikasi nya sering force close sendiri sudah...
9,8,1551,8_daftar_susah_saja susah_daftar saja,"[daftar, susah, saja susah, daftar saja, mau d...","[mau daftar saja susah, mau daftar saja susah ..."


In [37]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 180
Outliers            : 125,526
Outlier Percentage  : 64.76%


Topic Size

In [38]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,125526
1,0,16144
2,1,3854
3,2,3491
4,3,3369
...,...,...
176,175,51
177,176,51
178,177,51
179,178,51


Top Words

In [39]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("="*80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('saldo', np.float64(0.011447670618166742)), ('uang', np.float64(0.007770735916102233)), ('saya', np.float64(0.006596088732017602)), ('tapi', np.float64(0.0057865280496202385)), ('ada', np.float64(0.005608750200529432)), ('biaya', np.float64(0.00535165492441615)), ('ke', np.float64(0.004867324434933718)), ('potongan', np.float64(0.0048104220965215225)), ('saldo saya', np.float64(0.004791456937386636)), ('berkurang', np.float64(0.0047800628915554))]
Topic 1
[('jaringan', np.float64(0.03206974342441535)), ('sinyal', np.float64(0.0270171047233818)), ('merah', np.float64(0.025431279204321984)), ('bagus', np.float64(0.021671119098773534)), ('koneksi', np.float64(0.020474219408311577)), ('indikator', np.float64(0.019853712212151417)), ('internet', np.float64(0.018990255727330208)), ('lampu', np.float64(0.018423424771052223)), ('wifi', np.float64(0.01790612454504731)), ('merah terus', np.float64(0.017348448690065625))]
Topic 2
[('wajah', np.float64(0.06361877407676611)), ('verifikasi

Representative Reviews

In [40]:
representative_docs = topic_model.get_representative_docs()

for topic in representative_docs:

    if topic == -1:
        continue

    print("="*100)

    print(f"Topic {topic}")

    print()

    for doc in representative_docs[topic][:5]:

        print("-", doc)

    print()

Topic 0

- saya melakukan transaksi gagal tapi saldo saya hilang terpotong
- transaksi qris gagal tapi saldo kepotong
- transaksi gagal tapi saldo berkurang

Topic 1

- sering banget error indikator merah terus padahal jaringan bagus
- ini kenapa lampu indikator selalu merah padahal sinyal jaringan bagus
- lampu indikator selalu merah padahal sinyal bagus

Topic 2

- verifikasi wajah gagal terus enggak jelas
- verifikasi wajah gagal terus enggak jelas
- verifikasi wajah gagal mulu

Topic 3

- semenjak update aplikasi nya enggak bisa di buka tolong segera di perbaiki
- tolong di perbaiki
- tolong di perbaiki

Topic 4

- kalo sudah jam 23 00 sampai jam 03 00 enggak bisa buat transaksi
- selalu gagal top up di jam jam malam khususnya tengah malam
- selalu gangguan di jam 12 malam ke atas

Topic 5

- setelah di update aplikasi enggak bisa di buka kenapa ya
- kenapa setelah di update enggak bisa di buka
- kenapa setelah di update enggak bisa di buka

Topic 6

- enggak bisa login padahal use

silhoutte score

In [41]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.3900


In [42]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.6972


Representative Reviews

In [43]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():

    if topic == -1:
        continue

    print("="*100)

    print(f"TOPIC {topic}")

    print()

    for i, doc in enumerate(docs[:5],1):

        print(f"{i}. {doc}")

        print()

TOPIC 0

1. saya melakukan transaksi gagal tapi saldo saya hilang terpotong

2. transaksi qris gagal tapi saldo kepotong

3. transaksi gagal tapi saldo berkurang

TOPIC 1

1. sering banget error indikator merah terus padahal jaringan bagus

2. ini kenapa lampu indikator selalu merah padahal sinyal jaringan bagus

3. lampu indikator selalu merah padahal sinyal bagus

TOPIC 2

1. verifikasi wajah gagal terus enggak jelas

2. verifikasi wajah gagal terus enggak jelas

3. verifikasi wajah gagal mulu

TOPIC 3

1. semenjak update aplikasi nya enggak bisa di buka tolong segera di perbaiki

2. tolong di perbaiki

3. tolong di perbaiki

TOPIC 4

1. kalo sudah jam 23 00 sampai jam 03 00 enggak bisa buat transaksi

2. selalu gagal top up di jam jam malam khususnya tengah malam

3. selalu gangguan di jam 12 malam ke atas

TOPIC 5

1. setelah di update aplikasi enggak bisa di buka kenapa ya

2. kenapa setelah di update enggak bisa di buka

3. kenapa setelah di update enggak bisa di buka

TOPIC 6

1

NPMI

In [52]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [53]:
doc.split()

['bukannya', 'mempermudah', 'malah', 'mempersulit']

In [54]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [56]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [57]:
# sanity check 
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 180

[['saldo', 'uang', 'saya', 'tapi', 'ada', 'biaya', 'ke', 'potongan', 'saldo saya', 'berkurang'], ['jaringan', 'sinyal', 'merah', 'bagus', 'koneksi', 'indikator', 'internet', 'lampu', 'wifi', 'merah terus'], ['wajah', 'verifikasi wajah', 'verifikasi', 'gagal', 'wajah gagal', 'gagal terus', 'selalu gagal', 'susah', 'wajah selalu', 'verivikasi']]


In [58]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.0691
